# Train CNN
*This notebook aims to train a CNN model to classify images (pokemon 2D pictures)*

## Summary

- [Imports & Configuration](#imports--configuration)
- [Data Preparation](#data-preparation)
- [Model](#model)
- [Optimizer and Criterion](#optimizer-and-criterion)
- [Training with logging MLFlow](#training-with-loggin-mlflow)

## Imports & Configuration

In [13]:
import os
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
mlflow.set_experiment("final-project")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Data Preparation

In [14]:
from pathlib import Path
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split

# Recherche du dossier data/raw dans les parents
cwd = Path.cwd()
raw_dir = None
for ancestor in [cwd] + list(cwd.parents):
    candidate = ancestor / "data" / "raw"
    if candidate.is_dir():
        raw_dir = candidate
        break

if raw_dir is None:
    raise FileNotFoundError(f"Impossible de trouver le dossier 'data/raw' depuis {cwd}")

print(f"→ RAW_DIR détecté : {raw_dir}")

# Hyper-paramètres
BATCH_SIZE  = 32
IMG_SIZE    = 224
TRAIN_RATIO = 0.8
SEED        = 42

# Transforms
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

# Chargement complet
full_dataset = ImageFolder(str(raw_dir))
class_names  = full_dataset.classes
n_total      = len(full_dataset)
n_train      = int(n_total * TRAIN_RATIO)
n_val        = n_total - n_train

# Split aléatoire
torch.manual_seed(SEED)
train_subset, val_subset = random_split(full_dataset, [n_train, n_val])

# Assigner les transforms respectifs
train_subset.dataset.transform = train_transforms
val_subset.dataset.transform   = val_transforms

# Création des DataLoaders
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Total images: {n_total} → Train: {n_train}, Val: {n_val}")
print(f"Classes détectées: {class_names}")


→ RAW_DIR détecté : c:\Users\marco\Desktop\devops-pokefind\data\raw
Total images: 10657 → Train: 8525, Val: 2132
Classes détectées: ['Abra', 'Aerodactyl', 'Alakazam', 'Arbok', 'Arcanine', 'Articuno', 'Beedrill', 'Bellsprout', 'Blastoise', 'Bulbasaur', 'Butterfree', 'Caterpie', 'Chansey', 'Charizard', 'Charmander', 'Charmeleon', 'Clefable', 'Clefairy', 'Cloyster', 'Cubone', 'Dewgong', 'Diglett', 'Ditto', 'Dodrio', 'Doduo', 'Dragonair', 'Dragonite', 'Dratini', 'Drowzee', 'Dugtrio', 'Eevee', 'Ekans', 'Electabuzz', 'Electrode', 'Exeggcute', 'Exeggutor', 'Farfetchd', 'Fearow', 'Flareon', 'Gastly', 'Gengar', 'Geodude', 'Gloom', 'Golbat', 'Goldeen', 'Golduck', 'Golem', 'Graveler', 'Grimer', 'Growlithe', 'Gyarados', 'Haunter', 'Hitmonchan', 'Hitmonlee', 'Horsea', 'Hypno', 'Ivysaur', 'Jigglypuff', 'Jolteon', 'Jynx', 'Kabuto', 'Kabutops', 'Kadabra', 'Kakuna', 'Kangaskhan', 'Kingler', 'Koffing', 'Krabby', 'Lapras', 'Lickitung', 'Machamp', 'Machoke', 'Machop', 'Magikarp', 'Magmar', 'Magnemite', 'M

## Model

In [16]:
from torchvision import models

# Charger un ResNet18 pré-entraîné
model = models.resnet18(pretrained=True)

# Récupérer le nombre de features de la dernière couche
num_ftrs = model.fc.in_features

# Adapter la dernière couche au nombre de classes détectées
# class_names vient de la cellule 2 : full_dataset.classes
model.fc = nn.Linear(num_ftrs, len(class_names))

# Envoyer sur GPU/CPU
model = model.to(device)

print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Optimizer and Criterion

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


## Training with loggin MLFlow

In [18]:
EPOCHS = 5

with mlflow.start_run():
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("lr", 1e-3)
    mlflow.log_param("model_arch", "resnet18")

    for epoch in range(1, EPOCHS+1):
        # --- Training ---
        model.train()
        running_loss = 0.0
        correct = total = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        train_loss = running_loss / total
        train_acc  = correct / total

        # --- Validation ---
        model.eval()
        val_loss = val_correct = val_total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        val_loss /= val_total
        val_acc  = val_correct / val_total

        # --- Logging MLflow ---
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("train_acc",  train_acc,  step=epoch)
        mlflow.log_metric("val_loss",   val_loss,   step=epoch)
        mlflow.log_metric("val_acc",    val_acc,    step=epoch)

        print(f"Epoch {epoch}/{EPOCHS} • "
              f"Train: loss={train_loss:.4f}, acc={train_acc:.4f} • "
              f"Val:   loss={val_loss:.4f}, acc={val_acc:.4f}")

    # Sauvegarde du modèle final dans MLflow
    mlflow.pytorch.log_model(model, "model")
    print("Model logged to MLflow.")


Epoch 1/5 • Train: loss=2.8834, acc=0.3462 • Val:   loss=1.9470, acc=0.4944
Epoch 2/5 • Train: loss=1.2835, acc=0.6592 • Val:   loss=1.7007, acc=0.5750
Epoch 3/5 • Train: loss=0.7697, acc=0.7920 • Val:   loss=1.3487, acc=0.6609
Epoch 4/5 • Train: loss=0.4311, acc=0.8818 • Val:   loss=1.3876, acc=0.6656


2025/07/10 19:09:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 5/5 • Train: loss=0.2987, acc=0.9199 • Val:   loss=1.3674, acc=0.6778


2025/07/10 19:09:47 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Model logged to MLflow.
